In [0]:
%python
%pip install -r /Workspace/Users/shuailuan@gmail.com/meteor_api-LUAN/requirements.txt

In [0]:
%python
%pip install xmltodict pyhumps pydantic-settings aiohttp fastapi uvicorn

In [0]:
%python
dbutils.library.restartPython()

In [0]:
%python
import sys
sys.path.insert(0, '/Workspace/Users/shuailuan@gmail.com/meteor_api-LUAN')

# Test all the imports from your FastAPI application
from app import external_services as services
from app.config import Settings, get_settings
from app.models import (
    AllMaps,
    CurrentWeather,
    Forecast,
    IsobaricMaps,
    LegacyCurrentWeather,
    RainMaps,
)
from app.sessions import close_session, get_session

print("✅ All imports from __main__.py are working!")
print(f"✅ Settings: {Settings}")
print(f"✅ CurrentWeather: {CurrentWeather}")
print("\n🎉 Your FastAPI application is ready to run!")

In [0]:
%python
import sys
sys.path.insert(0, '/Workspace/Users/shuailuan@gmail.com/meteor_api-LUAN')

from app import wx_station

print("Testing wx_station module...\n")

# Test 1: Test the _degrees_to_cardinal helper function
print("Test 1: Cardinal direction conversion")
test_directions = [0, 45, 90, 135, 180, 225, 270, 315]
for degrees in test_directions:
    cardinal = wx_station._degrees_to_cardinal(degrees)
    print(f"  {degrees}° = {cardinal}")

print("\n" + "="*50 + "\n")

# Test 2: Test parse_current_wx_data with sample XML
print("Test 2: Parse weather XML data")

# Sample XML data matching the expected format (with multiple items)
sample_xml_str = '''<?xml version="1.0" encoding="UTF-8"?>
<rss version="2.0">
  <channel>
    <item>
      <description>Outside Temp: 22.5°C
Pressure: 1013.25 MBar
Wind: 12.5 knots from 270°;
Rain Rate: 0.0 mm/hr
Inside Temp: 21.0°C</description>
    </item>
    <item>
      <description>dummy</description>
    </item>
  </channel>
</rss>'''
sample_xml = sample_xml_str.encode('utf-8')

try:
    result = wx_station.parse_current_wx_data(sample_xml)
    print("✅ Successfully parsed weather data:")
    print(f"  Outside Temperature: {result['outside_temp']}°C")
    print(f"  Inside Temperature: {result['inside_temp']}°C")
    print(f"  Pressure: {result['pressure_MBar']} MBar")
    print(f"  Rain Rate: {result['rain_rate']} mm/hr")
    print(f"  Wind Speed: {result['wind']['speed_kts']} knots")
    print(f"  Wind Direction: {result['wind']['direction_degrees']}° ({result['wind']['cardinal_str']})")
except Exception as e:
    print(f"❌ Error parsing weather data: {e}")
    import traceback
    traceback.print_exc()

print("\n✅ wx_station module tests complete!")

In [0]:
%python
dbutils.library.restartPython()

In [0]:
%python
import sys
sys.path.insert(0, '/Workspace/Users/shuailuan@gmail.com/meteor_api-LUAN')

import asyncio
from aiohttp import ClientSession
from app import external_services
from app.config import Settings

print("Testing Christchurch Forecasts...\n")

async def test_forecasts():
    """Test fetching forecast data for Christchurch"""
    city = "christchurch"
    
    async with ClientSession() as session:
        try:
            # Fetch forecast data
            forecast = await external_services.get_forecasts(session, city)
            
            print(f"✅ Successfully fetched forecast data for {city.title()}!\n")
            print(f"Total forecast days: {len(forecast.days)}\n")
            print("=" * 70)
            
            # Display first 3 days in detail
            for i, day in enumerate(forecast.days[:3], 1):
                print(f"\nDay {i}: {day.dow} - {day.date}")
                print(f"  Temperature: Min {day.min}°C, Max {day.max}°C")
                print(f"  Forecast: {day.forecast_word}")
                print(f"  Details: {day.forecast}")
                print(f"  Issued at: {day.issued_at}")
                
                if day.rise_set:
                    print(f"  Sunrise: {day.rise_set.sun_rise}")
                    print(f"  Sunset: {day.rise_set.sun_set}")
                print("-" * 70)
            
            # Summary of remaining days
            if len(forecast.days) > 3:
                print(f"\n... and {len(forecast.days) - 3} more days\n")
                
            print("\n✅ Forecast test complete!")
            return forecast
            
        except Exception as e:
            print(f"❌ Error fetching forecast data: {e}")
            import traceback
            traceback.print_exc()
            return None

# Run the async test
forecast_result = await test_forecasts()